# Olist End-to-End Business Data Analytics
## Notebook 02 — Exploratory Data Analysis & Business Analysis

**Final Project — Data Analytics | Notebook 2 of 3**

The goal is to understand the main business patterns behind seller acquisition, marketplace performance and customer satisfaction before moving to SQL and final business recommendations.

## Main business questions

1. Which acquisition channels bring sellers with stronger commercial performance after conversion?
2. How reliable is delivery performance, and are there meaningful regional differences?
3. Which factors are most associated with lower customer satisfaction: delivery performance, product category, or price?
4. Do sellers acquired through different channels show different commercial performance, operational performance, and customer satisfaction?

## Analysis scope

Seller performance will be analyzed across three separate dimensions:

- **Commercial performance:** GMV as the main metric, supported by number of orders, AOV and observed active period.
- **Operational performance:** delivery time and delivery reliability.
- **Customer satisfaction:** review score.

A key limitation is that only 380 of the 842 Marketing-acquired sellers are observed in the available E-Commerce data. Any downstream seller-performance analysis therefore applies only to this observable subset.

In [1]:
# Import libraries 

import pandas as pd
import numpy as np

# for visualizations
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Path to cleaned datasets exported from Notebook 1

PROCESSED_PATH = "../data/processed"

In [3]:
# Load cleaned E-Commerce datasets

customers = pd.read_csv(f"{PROCESSED_PATH}/customers_clean.csv")
orders = pd.read_csv(f"{PROCESSED_PATH}/orders_clean.csv")
order_items = pd.read_csv(f"{PROCESSED_PATH}/order_items_clean.csv")
payments = pd.read_csv(f"{PROCESSED_PATH}/payments_clean.csv")
reviews = pd.read_csv(f"{PROCESSED_PATH}/reviews_clean.csv")
products = pd.read_csv(f"{PROCESSED_PATH}/products_clean.csv")
sellers = pd.read_csv(f"{PROCESSED_PATH}/sellers_clean.csv")
geolocation = pd.read_csv(f"{PROCESSED_PATH}/geolocation_clean.csv")
category_translation = pd.read_csv(f"{PROCESSED_PATH}/category_translation_clean.csv")

In [4]:
# Load cleaned Marketing Funnel datasets

mql = pd.read_csv(f"{PROCESSED_PATH}/mql_clean.csv")
closed_deals = pd.read_csv(f"{PROCESSED_PATH}/closed_deals_clean.csv")

In [5]:
# Load ZIP-level geolocation reference created in Notebook 1

geolocation_zip = pd.read_csv(f"{PROCESSED_PATH}/geolocation_zip_reference.csv")

In [6]:
print("E-COMMERCE DATASETS")
print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("geolocation:", geolocation.shape) 
print("category_translation:", category_translation.shape) 

print("\nMARKETING FUNNEL DATASETS")
print("mql:", mql.shape)
print("closed_deals:", closed_deals.shape) 

print("\nZIP-level geolocation ")
print("geolocation_zip:", geolocation_zip.shape)

E-COMMERCE DATASETS
customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 5)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (738332, 5)
category_translation: (71, 2)

MARKETING FUNNEL DATASETS
mql: (8000, 4)
closed_deals: (842, 14)

ZIP-level geolocation 
geolocation_zip: (19015, 3)


### Data loaded

The cleaned datasets exported from Notebook 01 were loaded successfully.

The analysis will keep the E-Commerce and Marketing Funnel datasets separate until a specific business question requires connecting them. The ZIP-level geolocation reference will be used only when geographic coordinates are needed.

## 1. Prepare the analytical variables 

## Business metrics dictionary

Before building the analytical dataset, the main business metrics used in this analysis are defined below. These definitions will be used consistently throughout the notebook.

| Metric | Definition | Calculation |
|---|---|---|
| **GMV** | Gross Merchandise Value. Total value of products sold by a seller in the marketplace.  | Sum of `price` |
| **Number of orders** | Number of unique orders in which a seller participated. | Unique count of `order_id` |
| **AOV** | Average value generated by the seller per order. | GMV / Number of unique orders |
| **Observed active period** | Time between the seller's first and last observed sale in the available dataset. | Last purchase date - First purchase date |
| **Average delivery time** | Average number of days between purchase and delivery to the customer. | Delivery date - Purchase date |
| **Late delivery rate** | Percentage of delivered orders that arrived after the estimated delivery date. | Late delivered orders / Delivered orders |
| **Delivery time variability** | Variation in delivery time across the seller's orders, used as an indicator of delivery consistency. | Standard deviation of delivery days |
| **Average review score** | Average customer review score associated with the seller's orders. | Mean `review_score` |

### 1.1. Build the seller-level analytical dataset

The seller-level analytical dataset will contain one row per seller and will bring together the main metrics needed for later business analysis.

The first step is to build the commercial performance metrics from `order_items`.

### Commercial performance

In [7]:
seller_commercial = order_items.groupby("seller_id").agg(gmv=("price", "sum"), number_of_orders=("order_id", "nunique")).reset_index() 
seller_commercial.head()

,seller_id,gmv,number_of_orders
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200
2,001e6ad469a905060d959994f1b41e4f,250.00,1
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51
4,003554e2dce176b5555353e4f3555ac8,120.00,1


In [8]:
#AOV at the seller level: GMV / number of unique orders in which they participated.

seller_commercial["aov"] = seller_commercial["gmv"] / seller_commercial["number_of_orders"]
seller_commercial.head()

# Note: This is the value of what THAT seller sold within the order, not the total value of the customer's order (which may include other sellers).

,seller_id,gmv,number_of_orders,aov
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3,895.000000
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200,125.400150
2,001e6ad469a905060d959994f1b41e4f,250.00,1,250.000000
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51,24.205882
4,003554e2dce176b5555353e4f3555ac8,120.00,1,120.000000


### Operational performance

### Customer satisfaction

###  1.2 Add Marketing acquisition information